# 10 - Documentation tables

**Goal:** write the numbers and tables used by the LaTeX documentation (`docs/latex/project_documentation.tex`) straight from the project's outputs, so the document cannot drift from the data.

**Inputs:** `data/processed/*.csv` (notebooks 03-07) and the packaged Tableau workbook `dashboard/tableau/retail_dashboard.twbx` (read directly from the zip, so hand edits made in Tableau are picked up).
**Output:** LaTeX fragments in `docs/latex/generated/`, pulled into the document with `\input{...}`:

| Fragment | Used for |
|---|---|
| `kpi_facts.tex`, `tableau_facts.tex` | Macros such as `\kpiNewDrop` and `\twbxBytes` used in the text |
| `rfm_table.tex`, `churn_bands_table.tex`, `rules_table.tex`, `recs_table.tex` | Analysis tables |
| `tableau_sources_table.tex`, `tableau_worksheets_table.tex`, `tableau_dashboards_table.tex` | Tableau workbook inventory |

Run this notebook after notebooks 03-09 (and after saving the workbook in Tableau), then recompile the document.

In [1]:
import io
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent  # notebook runs from notebooks/
PROC = ROOT / "data" / "processed"
TWBX = ROOT / "dashboard" / "tableau" / "retail_dashboard.twbx"
OUT = ROOT / "docs" / "latex" / "generated"
OUT.mkdir(parents=True, exist_ok=True)

ROW_END = r" \\"
SPECIAL = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#", "_": r"\_",
           "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}

def esc(text):
    """Escape text for LaTeX. Fails loudly on characters the document's fonts may not support."""
    out = "".join(SPECIAL.get(ch, ch) for ch in str(text))
    out = out.replace("£", r"\pounds{}").replace(">", "$>$").replace("<", "$<$")
    assert out.isascii(), f"non-ASCII character in {out!r}"
    return out

def money(v):
    return f"{v:,.0f}"

def kfmt(v):
    return rf"\pounds{{}}{v / 1e3:,.0f}K"

def pct(x, digits=1):
    return rf"{x * 100:.{digits}f}\%"

def table(colspec, header, rows, env="tabular", width=None):
    begin = rf"\begin{{{env}}}{{{width}}}{{{colspec}}}" if width else rf"\begin{{{env}}}{{{colspec}}}"
    lines = [begin, r"\toprule", " & ".join(header) + ROW_END, r"\midrule"]
    lines += [" & ".join(r) + ROW_END for r in rows]
    lines += [r"\bottomrule", rf"\end{{{env}}}"]
    return "\n".join(lines)

written = {}
def write(name, body):
    (OUT / f"{name}.tex").write_text(body.strip() + "\n", encoding="utf-8")
    written[name] = len(body.splitlines())

## 1. Analysis tables

In [2]:
rfm = pd.read_csv(PROC / "rfm_segments.csv").sort_values("revenue", ascending=False)
write("rfm_table", table(
    "lrrrr", ["Segment", "Customers", r"\% customers", r"Net revenue (\pounds)", r"\% revenue"],
    [[esc(r.segment), f"{r.customers:,}", f"{r.customer_share * 100:.1f}", money(r.revenue), f"{r.revenue_share * 100:.1f}"] for r in rfm.itertuples()],
))

bands = pd.read_csv(PROC / "churn_by_recency.csv")
write("churn_bands_table", table(
    "lrr", ["Days silent at cutoff", "Customers", "Churn rate"],
    [[esc(r.days_silent).replace("-", "--"), f"{r.customers:,}", pct(r.churn_rate)] for r in bands.itertuples()],
))

backtest = pd.read_csv(PROC / "churn_rule_backtest.csv").set_index("rule")
SELECTED = ["silent > 60 days", "silent > 90 days", "silent > 120 days", "silent > 1.5x own avg gap (3+ orders)"]
write("rules_table", table(
    "lrrrr", ["Rule", "Flagged", "Precision", "Recall", "Churner revenue caught"],
    [[esc(rule), f"{backtest.loc[rule, 'flagged']:,}", pct(backtest.loc[rule, "precision"]), pct(backtest.loc[rule, "recall"]), pct(backtest.loc[rule, "revenue_captured"])]
     for rule in SELECTED],
))

recs = pd.read_csv(PROC / "recommendations.csv")
write("recs_table", table(
    "lXrrrr", [r"\#", "Action", "Base", "Low", "Mid", "High"],
    [[str(r.rank), esc(r.recommendation), kfmt(r.base_gbp), kfmt(r.low_gbp), kfmt(r.mid_gbp), kfmt(r.high_gbp)] for r in recs.itertuples()],
    env="tabularx", width=r"\textwidth",
))
written

{'rfm_table': 16, 'churn_bands_table': 12, 'rules_table': 10, 'recs_table': 10}

## 2. The packaged Tableau workbook

Everything below is read from the `.twbx` itself, so it describes the file that is actually presented, including the sheets added by hand in Tableau.

In [3]:
z = zipfile.ZipFile(TWBX)
twb_name = next(n for n in z.namelist() if n.endswith(".twb"))
root = ET.fromstring(z.read(twb_name))
csvs = {n.split("/")[-1]: pd.read_csv(io.BytesIO(z.read(n))) for n in z.namelist() if n.endswith(".csv")}

caption_by_file = {}
for ds in root.find("datasources"):
    conn = ds.find(".//connection[@class='textscan']")
    caption_by_file[conn.get("filename")] = ds.get("caption")

sheets = []
for ws in root.find("worksheets"):
    sheets.append((ws.get("name"), ws.find("table/panes/pane/mark").get("class"), ws.find("table/view/datasources/datasource").get("caption")))
tiles = [s for s in sheets if s[0].startswith("T ")]
charts = [s for s in sheets if not s[0].startswith("T ")]

dashboards = []
for d in root.find("dashboards"):
    size = d.find("size")
    names = {zn.get("name") for zn in d.iter("zone") if zn.get("name")}
    banner = next(
        "".join(next(zn.iter("run")).itertext()).replace("Æ", "").strip()
        for zn in d.iter("zone") if zn.get("type-v2") == "text"
    )
    dashboards.append((d.get("name"), f"{size.get('minwidth')}$\\times${size.get('minheight')}", len(names), banner))
print(len(sheets), "worksheets:", len(tiles), "tiles +", len(charts), "charts/tables;", len(dashboards), "dashboards;", len(csvs), "data files")
dashboards

25 worksheets: 12 tiles + 13 charts/tables; 5 dashboards; 12 data files


[('Actions',
  '1280$\\times$650',
  1,
  'Four moves, sized under low, mid and high scenarios'),
 ('Churn risk',
  '1280$\\times$1040',
  7,
  'Silence is the signal: contact customers quiet for 90+ days, biggest first'),
 ('Overview',
  '1280$\\times$1040',
  8,
  'Sales are flat while new-customer acquisition fell 41%'),
 ('Retention',
  '1280$\\times$1090',
  3,
  'Cohort quality is stable since mid-2010; acquisition volume is what fell'),
 ('Segments', '1280$\\times$990', 5, '24% of customers bring 70% of revenue')]

In [4]:
# data sources
write("tableau_sources_table", table(
    "lrrl", ["File", "Rows", "Columns", "Data source"],
    [[rf"\file{{{fn}}}", f"{len(df):,}", str(df.shape[1]), esc(caption_by_file.get(fn, "-"))] for fn, df in csvs.items()],
))

# worksheets (the 12 tile sheets are summarised in the text; charts and tables are listed here)
PURPOSE = {
    "Monthly sales": "Gross sales by month (kept in the workbook; not on the packaged Overview)",
    "Monthly sales (2)": "Gross sales by calendar month, 2011 against 2010 (added in Tableau)",
    "New vs returning": "Existing and new customers per month",
    "New vs returning (2)": "New customers by calendar month, 2011 against 2010 (added in Tableau)",
    "Cohort heatmap": "Retention by cohort and month",
    "Repeat 90d": "90-day repeat rate by acquisition period",
    "New customers by region": "New customers, domestic and overseas",
    "RFM segments": "Customer and revenue shares by segment",
    "Concentration": "Cumulative share of sales by share of customers",
    "Churn by silence": "Churn rate by days silent",
    "Rule back-test": "Precision, recall and revenue caught per rule",
    "Contact list": "Top-value customers to contact first",
    "Recommendations": "Sized recommendations",
}
unknown = [c[0] for c in charts if c[0] not in PURPOSE]
rows = [[esc(n), esc(m), esc(PURPOSE.get(n, "(describe in notebook 10)"))] for n, m, _ in sorted(charts, key=lambda c: list(PURPOSE).index(c[0]) if c[0] in PURPOSE else 99)]
write("tableau_worksheets_table", table("llp{6.3cm}", ["Worksheet", "Mark", "Purpose"], rows))

# dashboards
write("tableau_dashboards_table", table(
    "llrX", ["Dashboard", "Size", "Sheets", "Banner headline"],
    [[esc(n), s, str(k), r"``" + esc(b) + r"''"] for n, s, k, b in dashboards],
    env="tabularx", width=r"\textwidth",
))
{"charts without a purpose text": unknown}

{'charts without a purpose text': []}

## 3. Macros for numbers used in the text

In [5]:
def macros(name, values):
    body = "\n".join(rf"\newcommand{{\{k}}}{{{v}}}" for k, v in values.items())
    write(name, "% generated by notebooks/10_documentation_tables.ipynb - do not edit\n" + body)

kp = csvs["headline_kpis.csv"].iloc[0]
champ = rfm.set_index("segment").loc["Champions"]
macros("kpi_facts", {
    "kpiGross": money(kp.gross_sales),
    "kpiTracked": f"{kp.tracked_customers:,.0f}",
    "kpiRepeat": f"{kp.repeat_rate * 100:.0f}",
    "kpiYoy": f"{kp.sales_growth * 100:.1f}",
    "kpiNewDrop": f"{abs(kp.new_customer_change) * 100:.0f}",
    "kpiChampCust": f"{champ.customer_share * 100:.0f}",
    "kpiChampRev": f"{champ.revenue_share * 100:.0f}",
    "kpiFlagged": f"{kp.flagged_customers:,.0f}",
    "kpiFlaggedRev": money(kp.flagged_sales),
    "kpiExpectedLoss": money(kp.expected_loss),
})
macros("tableau_facts", {
    "twbxBytes": f"{TWBX.stat().st_size:,}",
    "twbBytes": f"{z.getinfo(twb_name).file_size:,}",
    "tabBuild": esc(root.get("source-build")),
    "nSheets": len(sheets), "nTiles": len(tiles), "nCharts": len(charts),
    "nDashboards": len(dashboards), "nDataFiles": len(csvs),
})
pd.Series(written, name="lines").to_frame()

,lines
rfm_table,16
churn_bands_table,12
rules_table,10
recs_table,10
tableau_sources_table,18
tableau_worksheets_table,19
tableau_dashboards_table,11
kpi_facts,11
tableau_facts,9


## 4. Robustness tables

Tables for the *Robustness checks* section of the document, read from the outputs of notebook 11 (run notebook 11 first).

In [6]:
def num(x, digits=1, sign=False):
    text = f"{x:+.{digits}f}" if sign else f"{x:.{digits}f}"
    return text.replace("-", "$-$")

cut = pd.read_csv(PROC / "robustness_cutoffs.csv")
write("robustness_cutoffs_table", table(
    "lrrrrrr", ["Cutoff", "Base", "Churn rate", r"Precision ($>$ 90 d)", r"Recall ($>$ 90 d)", "Revenue caught", "Own-gap lift"],
    [[r.cutoff, f"{r.base:,}", pct(r.churn_rate), pct(r.p90), pct(r.r90), pct(r.rev90), f"{r.gap_rule_precision / r.gap_rule_base_rate:.2f}"] for r in cut.itertuples()],
))

iv = pd.read_csv(PROC / "robustness_intervals.csv")
def short(method):
    if method.startswith("cluster bootstrap"):
        return "bootstrap"
    if method.startswith("exact binomial"):
        return "exact binomial"
    return method.replace(" interval", "")
SIGNED = ("Sales growth", "New customers")
write("robustness_intervals_table", table(
    "Xrrl", ["Metric", "Estimate", r"95\% interval", "Method"],
    [[esc(r.metric), num(r.estimate, 1, sign=r.metric.startswith(SIGNED)) + r"\%",
      num(r.ci_low, 1, sign=r.metric.startswith(SIGNED)) + r"\% to " + num(r.ci_high, 1, sign=r.metric.startswith(SIGNED)) + r"\%", short(r.method)]
     for r in iv.itertuples()],
    env="tabularx", width=r"\textwidth",
))

cmp_ = pd.read_csv(PROC / "robustness_model_comparison.csv", index_col=0)
def fmt_metric(name, v):
    return f"{v:,.0f}" if name == "AIC" else (f"{v:,.1f}" if name == "log-likelihood" else f"{v:.3f}")
write("product_model_table", table(
    "lrr", ["Measure", "Baseline", "With product features"],
    [[esc(name), fmt_metric(name, row["baseline"]).replace("-", "$-$"), fmt_metric(name, row["with product features"]).replace("-", "$-$")] for name, row in cmp_.iterrows()],
))
cmp_

,baseline,with product features
log-likelihood,-2369.985625,-2360.122033
AIC,4751.971250,4742.244067
pseudo R2,0.199169,0.202502
in-sample AUC,0.785560,0.788153
out-of-time AUC,0.777799,0.777632
